# Document Processing for Knowledge Assistant: Reference Patterns

This notebook is a reference for the building blocks needed to take raw ingested documents and turn them into a Knowledge Assistant-ready corpus. Each section is a self-contained worked example for one specific capability:

- Landing raw documents
- Filtering for relevance
- Detecting exact and near-duplicate content
- Staging documents for human review and approval
- Parsing with `ai_parse_document`
- Section-grouped chunking
- Metadata extraction with `ai_extract`
- Handing off to Knowledge Assistant

Each capability can be used independently. Lift the SQL into your own pipeline as needed. The sections are sequenced as they would appear in a typical end-to-end pipeline, but any subset works.

All AI SQL functions referenced here are GA and HIPAA-eligible on workspaces with the Compliance Security Profile enabled.

This notebook is designed as a reference, not as a turnkey pipeline. The code uses placeholders throughout. Update them to match your environment before running.

## Prerequisites

- Databricks workspace with Unity Catalog enabled and serverless compute on.
- Destination catalog and schema where landing, staging, and gold tables will live.
- For HIPAA workloads, the workspace must have the Enhanced Security and Compliance add-on enabled.
- AI SQL functions enabled in your workspace (default for most recent workspaces).

## Assumed data model

This notebook assumes a medallion-style flow:

- **Bronze** (raw documents): documents land here as-is from any source. One row per document.
- **Staging**: documents that have passed relevance filtering and await human approval.
- **Silver** (parsed): approved documents with their structure extracted by `ai_parse_document`.
- **Gold** (chunks): section-grouped chunks with metadata, ready for Knowledge Assistant.

You do not need to use all four tiers. Skip whichever ones do not apply to your use case.

## Placeholders used in this notebook

| Placeholder | Description |
|---|---|
| `<CATALOG>` | Destination Unity Catalog |
| `<SCHEMA>` | Destination schema |
| `<RAW_TABLE>` | Bronze table holding raw documents |
| `<STAGING_TABLE>` | Staging table for human review |
| `<PARSED_TABLE>` | Silver table with `ai_parse_document` output |
| `<CHUNKS_TABLE>` | Gold table of section-grouped chunks |

## Capability 1: Landing raw documents

The first step is to get documents into a Delta table where downstream processing can act on them. The schema below is intentionally simple: a content column for raw bytes, a metadata column for source attribution, and a discovered_at timestamp for tracking when the document entered the system.

If your source is SharePoint, populate this table using the Lakeflow Connect SharePoint connector (see `01-sharepoint-connection-setup.ipynb` in this repo). If your source is an external crawler or RSS feed, write a Python notebook or Lakeflow Job that fetches the content and inserts into this table.

In [0]:
-- Capability 1: Bronze landing table schema
-- Use the same shape regardless of source (SharePoint, web crawl, manual upload).

CREATE TABLE IF NOT EXISTS <CATALOG>.<SCHEMA>.<RAW_TABLE> (
  doc_id          STRING,
  source_url      STRING,
  title           STRING,
  content         BINARY,
  content_type    STRING,
  source          STRING       COMMENT 'e.g., sharepoint, external_crawler, manual',
  discovered_at   TIMESTAMP
)
USING DELTA
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
COMMENT 'Bronze landing table for raw documents from any source';

## Capability 2: Filter for relevance

Once content is landing in the bronze table, the next step is to score each document for relevance to the chatbot's domain. Documents below a relevance threshold are dropped or set aside; documents above the threshold continue to the next stage.

Two patterns:

- `ai_classify` for known categories (relevant vs irrelevant, or finer-grained labels). Cleaner output, easier to reason about.
- `ai_query` for nuanced scoring with a custom prompt (e.g., a 1-to-10 score with explanation).

Both run on serverless SQL. For large bronze tables, batch with `LIMIT` or filter to recently-discovered documents to control cost.

In [0]:
-- Capability 2: Score each new bronze document for relevance.
-- Two patterns shown side-by-side; pick whichever fits your use case.
-- Customize the categories or prompt for your domain.

-- Pattern A: Categorical with ai_classify
SELECT
  doc_id,
  title,
  ai_classify(
    CAST(content AS STRING),
    ARRAY('relevant', 'irrelevant')
  ) AS relevance_label
FROM <CATALOG>.<SCHEMA>.<RAW_TABLE>
WHERE discovered_at >= current_date() - INTERVAL 7 DAYS
LIMIT 100;

-- Pattern B: Numeric score with ai_query (more nuanced)
SELECT
  doc_id,
  title,
  CAST(
    ai_query(
      'databricks-claude-sonnet-4-6',
      CONCAT(
        'Rate this article 1 to 10 on its relevance to your domain. ',
        'Return only the number, no other text. Article: ',
        CAST(content AS STRING)
      )
    ) AS INT
  ) AS relevance_score
FROM <CATALOG>.<SCHEMA>.<RAW_TABLE>
WHERE discovered_at >= current_date() - INTERVAL 7 DAYS
LIMIT 100;

## Capability 3: Detect duplicates and near-duplicates

This is the "tunnel vision" guard. Two layers:

**Exact duplicates** — identical content republished across multiple sources. Detected via a content hash. Cheap, fast, deterministic.

**Near-duplicates** — semantically identical but different wording (e.g., two articles summarizing the same press release). Detected via `ai_similarity`, which compares two strings and returns a 0-to-1 similarity score.

For very large corpora (10M+ docs), the all-pairs `ai_similarity` query becomes expensive. Switch to a Vector Search nearest-neighbor approach in that case: embed every doc into a vector index, then for each new doc query for its top-k nearest neighbors and check the cosine distance.

In [0]:
-- Capability 3a: Exact-duplicate detection via content hash

ALTER TABLE <CATALOG>.<SCHEMA>.<RAW_TABLE>
  ADD COLUMNS IF NOT EXISTS (content_sha256 STRING);

UPDATE <CATALOG>.<SCHEMA>.<RAW_TABLE>
SET content_sha256 = sha2(content, 256)
WHERE content_sha256 IS NULL;

-- Surface exact duplicates for review
SELECT
  content_sha256,
  COUNT(*) AS dup_count,
  COLLECT_LIST(doc_id) AS doc_ids
FROM <CATALOG>.<SCHEMA>.<RAW_TABLE>
GROUP BY content_sha256
HAVING COUNT(*) > 1
ORDER BY dup_count DESC;

-- Capability 3b: Near-duplicate detection via ai_similarity
-- Compares every pair once (a.doc_id < b.doc_id) to avoid mirror duplicates.
-- Adjust the 0.92 threshold based on your tolerance.

WITH pairs AS (
  SELECT
    a.doc_id AS doc_id_a,
    b.doc_id AS doc_id_b,
    a.title  AS title_a,
    b.title  AS title_b,
    ai_similarity(
      CAST(a.content AS STRING),
      CAST(b.content AS STRING)
    ) AS similarity
  FROM <CATALOG>.<SCHEMA>.<RAW_TABLE> a
  JOIN <CATALOG>.<SCHEMA>.<RAW_TABLE> b ON a.doc_id < b.doc_id
  WHERE a.content_sha256 != b.content_sha256
)
SELECT *
FROM pairs
WHERE similarity > 0.92
ORDER BY similarity DESC
LIMIT 100;

## Capability 4: Stage documents for human review

Even with automated relevance filtering, sensitive domains (regulations, governance policies) usually need a human in the loop to approve what enters the chatbot's knowledge base. The pattern is a staging Delta table with an `approved` boolean flag.

Documents land here after passing relevance filtering. A reviewer flips `approved` to TRUE on rows that should be ingested. Only approved rows flow downstream.

The reviewer experience can be:

- A Databricks App with a UI showing one document at a time plus approve/reject buttons. Best UX, takes a day or two to build.
- A Databricks dashboard with a filter widget, where reviewers update the `approved` column manually via SQL. Faster to set up but clunkier.
- A spreadsheet export/import flow, where reviewers approve outside Databricks and approved `doc_id`s get UPDATEd back. Works but loses audit trail.

Pick whichever fits the team's appetite.

In [0]:
-- Capability 4: Staging table with human approval flag

CREATE TABLE IF NOT EXISTS <CATALOG>.<SCHEMA>.<STAGING_TABLE> (
  doc_id            STRING,
  title             STRING,
  source_url        STRING,
  content_preview   STRING       COMMENT 'First few hundred chars for quick review',
  relevance_score   DOUBLE,
  staged_at         TIMESTAMP,
  reviewed_by       STRING       COMMENT 'Email or username of the reviewer',
  reviewed_at       TIMESTAMP,
  approved          BOOLEAN,
  rejection_reason  STRING
)
USING DELTA
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
COMMENT 'Documents pending or completed human review';

-- Move relevance-passed documents into staging
INSERT INTO <CATALOG>.<SCHEMA>.<STAGING_TABLE>
SELECT
  doc_id,
  title,
  source_url,
  LEFT(CAST(content AS STRING), 500) AS content_preview,
  NULL AS relevance_score,  -- replace with your actual relevance score column
  current_timestamp() AS staged_at,
  NULL AS reviewed_by,
  NULL AS reviewed_at,
  NULL AS approved,
  NULL AS rejection_reason
FROM <CATALOG>.<SCHEMA>.<RAW_TABLE>
WHERE doc_id NOT IN (SELECT doc_id FROM <CATALOG>.<SCHEMA>.<STAGING_TABLE>);

-- Reviewer action: approve a document
-- UPDATE <CATALOG>.<SCHEMA>.<STAGING_TABLE>
--   SET approved = TRUE,
--       reviewed_by = '<reviewer-email>',
--       reviewed_at = current_timestamp()
--   WHERE doc_id = '<doc-id>';

-- Reviewer action: reject a document
-- UPDATE <CATALOG>.<SCHEMA>.<STAGING_TABLE>
--   SET approved = FALSE,
--       reviewed_by = '<reviewer-email>',
--       reviewed_at = current_timestamp(),
--       rejection_reason = 'off-topic'
--   WHERE doc_id = '<doc-id>';

## Capability 5: Parse documents with ai_parse_document

For approved documents, the next step is to parse them into structured elements. `ai_parse_document` handles PDFs, DOCX, PPTX, and images, returning a VARIANT with the document's structure: headings, paragraphs, tables, lists, bounding boxes, and page numbers.

This is GA and HIPAA-eligible on Compliance Security Profile workspaces. Use this in place of `ai_prep_search` (which is currently in Beta and not HIPAA-eligible).

In [0]:
-- Capability 5: Parse approved documents
-- Joins raw content with staging.approved=TRUE.

CREATE TABLE IF NOT EXISTS <CATALOG>.<SCHEMA>.<PARSED_TABLE> (
  doc_id      STRING,
  parsed_doc  VARIANT,
  page_count  INT,
  parsed_at   TIMESTAMP
)
USING DELTA
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
COMMENT 'Documents parsed by ai_parse_document, ready for chunking';

INSERT INTO <CATALOG>.<SCHEMA>.<PARSED_TABLE>
SELECT
  r.doc_id,
  ai_parse_document(r.content) AS parsed_doc,
  CAST(ai_parse_document(r.content):document:metadata:page_count AS INT) AS page_count,
  current_timestamp() AS parsed_at
FROM <CATALOG>.<SCHEMA>.<RAW_TABLE> r
JOIN <CATALOG>.<SCHEMA>.<STAGING_TABLE> s ON r.doc_id = s.doc_id
WHERE s.approved = TRUE
  AND r.doc_id NOT IN (SELECT doc_id FROM <CATALOG>.<SCHEMA>.<PARSED_TABLE>);

## Capability 6: Section-grouped chunking

Knowledge Assistant performs default chunking when you point it at a Delta table of raw documents. For documents with strong structural patterns (regulations, policies, technical specs), section-grouped chunking produces better retrieval quality than default character-based chunking.

The query below groups every parsed element under its parent heading. Each chunk is one section, with its heading and page range preserved as columns.

In [0]:
-- Capability 6: Section-grouped chunking
-- One row per section, with heading and page range preserved.

CREATE TABLE IF NOT EXISTS <CATALOG>.<SCHEMA>.<CHUNKS_TABLE> (
  chunk_id          STRING,
  doc_id            STRING,
  section_heading   STRING,
  chunk_text        STRING,
  start_page        INT,
  end_page          INT,
  chunked_at        TIMESTAMP
)
USING DELTA
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
COMMENT 'Section-grouped chunks ready for Knowledge Assistant';

INSERT INTO <CATALOG>.<SCHEMA>.<CHUNKS_TABLE>
WITH elements AS (
  SELECT
    doc_id,
    pos AS element_idx,
    element:type::STRING AS element_type,
    element:content::STRING AS element_text,
    CAST(element:bbox[0]:page_id AS INT) AS page_id
  FROM <CATALOG>.<SCHEMA>.<PARSED_TABLE>
  LATERAL VIEW posexplode(
    CAST(parsed_doc:document:elements AS ARRAY<VARIANT>)
  ) elem_table AS pos, element
  WHERE doc_id NOT IN (SELECT doc_id FROM <CATALOG>.<SCHEMA>.<CHUNKS_TABLE>)
),
sectioned AS (
  SELECT
    *,
    LAST_VALUE(
      CASE WHEN element_type LIKE 'heading%' THEN element_text END, true
    ) OVER (PARTITION BY doc_id ORDER BY element_idx) AS section_heading,
    LAST_VALUE(
      CASE WHEN element_type LIKE 'heading%' THEN element_idx END, true
    ) OVER (PARTITION BY doc_id ORDER BY element_idx) AS section_start_idx
  FROM elements
)
SELECT
  CONCAT(doc_id, '#sec-', CAST(COALESCE(section_start_idx, 0) AS STRING)) AS chunk_id,
  doc_id,
  section_heading,
  CONCAT_WS('\n\n', COLLECT_LIST(element_text)) AS chunk_text,
  MIN(page_id) AS start_page,
  MAX(page_id) AS end_page,
  current_timestamp() AS chunked_at
FROM sectioned
WHERE element_type NOT LIKE 'heading%'
  AND element_text IS NOT NULL
  AND LENGTH(element_text) > 20
GROUP BY doc_id, section_start_idx, section_heading
HAVING LENGTH(CONCAT_WS('\n\n', COLLECT_LIST(element_text))) > 100;

## Capability 7: Extract structured metadata per chunk

If retrieval needs to filter by attributes (effective date, issuing agency, topic, document type), extract those fields per chunk with `ai_extract`. The schema is user-defined; specify the fields you care about.

Skip this if your retrieval does not need structured filtering.

In [0]:
-- Capability 7: Enrich chunks with structured metadata
-- Adjust the schema below to match the fields relevant to your documents.

ALTER TABLE <CATALOG>.<SCHEMA>.<CHUNKS_TABLE>
  ADD COLUMNS IF NOT EXISTS (
    topic                  STRING,
    effective_date         DATE,
    issuing_agency         STRING,
    document_type          STRING,
    one_sentence_summary   STRING
  );

UPDATE <CATALOG>.<SCHEMA>.<CHUNKS_TABLE> AS target
SET (topic, effective_date, issuing_agency, document_type, one_sentence_summary) = (
  SELECT
    src.extracted:topic::STRING,
    CAST(src.extracted:effective_date AS DATE),
    src.extracted:issuing_agency::STRING,
    src.extracted:document_type::STRING,
    src.extracted:one_sentence_summary::STRING
  FROM (
    SELECT
      chunk_id,
      ai_extract(
        chunk_text,
        '{
          "topic": "STRING",
          "effective_date": "DATE",
          "issuing_agency": "STRING",
          "document_type": "STRING",
          "one_sentence_summary": "STRING"
        }'
      ) AS extracted
    FROM <CATALOG>.<SCHEMA>.<CHUNKS_TABLE>
    WHERE topic IS NULL
  ) AS src
  WHERE src.chunk_id = target.chunk_id
)
WHERE topic IS NULL;

## Capability 8: Hand off to Knowledge Assistant

Knowledge Assistant in Agent Bricks consumes a Delta table of pre-chunked text plus optional metadata columns for filtered retrieval. Embedding and indexing happen inside KA; no manual embedding step required.

Configuration steps:

1. In the Databricks left sidebar, open **Agent Bricks**.
2. Click **Knowledge Assistant** > **Create new agent**.
3. Set the knowledge source to the chunks table (`<CATALOG>.<SCHEMA>.<CHUNKS_TABLE>`).
4. Map the text column to `chunk_text`.
5. Map metadata columns (topic, effective_date, issuing_agency, etc.) as filter-eligible.
6. Configure a system prompt that scopes the agent's behavior to the corpus.
7. Click **Deploy**.

The deployed agent is accessible via the Databricks UI and reachable as a serving endpoint that can be embedded in Microsoft Teams or called from a custom Databricks App.

## Extension ideas

- **Auto-discovery from external sources.** Build a Lakeflow Job in Python that fetches new content from RSS feeds or sanctioned external sites on a schedule, lands raw content in the bronze table, and triggers the rest of the pipeline.
- **Review UI as a Databricks App.** A small Streamlit or React app that reads pending rows from the staging table, displays content previews, and offers approve/reject buttons. Much better UX than direct SQL updates.
- **Lakeflow Pipeline orchestration.** Wire each capability above as a step in a Lakeflow Pipeline so the full flow runs incrementally on a schedule.
- **Audit dashboard.** Build an AI/BI dashboard on the staging and chunks tables to monitor discovery rate, approval rate, dedup hit rate, and chunk quality over time.

## References

- [ai_parse_document](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_parse_document)
- [ai_extract](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_extract)
- [ai_classify](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_classify)
- [ai_query](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_query)
- [ai_similarity](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_similarity)
- [Knowledge Assistant](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/knowledge-assistant)
- [Vector Search](https://docs.databricks.com/aws/en/vector-search/vector-search)